In [1]:
import jax.numpy as jnp 
from chex import Array
from flax import linen as nn 
import optax 
import numpy as np 
from bandit import * 
from tqdm import tqdm 

from reinforced_lib import RLib 
from reinforced_lib.agents.deep import DQN

In [2]:
class QNetwork(nn.Module):
    n_actions: int = 5 

    @nn.compact
    def __call__(self, x: Array) -> Array: 

        x = nn.Dense(64)(x)
        x = nn.relu(x)

        x = nn.Dense(64)(x)
        x = nn.relu(x)

        q_values = nn.Dense(self.n_actions)(x)
        return q_values

In [ ]:
rl = RLib(
    agent_type = DQN,

    agent_params = {
        "q_network": QNetwork(n_actions=5), 

        "obs_space_shape": (2, ), 
        "act_space_size": 5, 

        "optimizer": optax.adam(1e-3), 

        "experience_replay_buffer_size": 1000,
        "experience_replay_batch_size": 32,
        "experience_replay_steps": 1,

        #contextual Bandit type
        "discount": 0.0, 

        "epsilon": 1.0, 
        "epsilon_decay": 0.995,
        "epsilon_min": 0.05
    },

    no_ext_mode = True
)

# hey - epsilon decay

In [4]:
rng = np.random.default_rng(42)

# IMPORTANT:
# Reinforced-lib initializes prev_env_state to zero.
# Therefore, make our actual initial state zero too.

state = np.zeros(2, dtype=np.float32)

# get initial action without performing a learning update 
action = rl.sample(
    is_training=False,
    sample_observations={
        "env_state": state,
    },
)

action = int(np.asarray(action))


ValueError: p must be None or a 1D vector with the same size as a.shape[axis]. p has shape (3,) and a.shape[axis] is 5.

In [ ]:
n_steps = 10_000
count = 0

for step in tqdm(range(n_steps)):
    # environment produces a reward for the previous action 
    reward = get_reward(rng, state, action)

    # the above reward is for the pervious state 
    next_state = sample_state(rng)

    next_action = rl.sample(
        update_observations={
            "env_state": next_state, 
            "action": action, 
            "reward": reward, 
            "terminal": False, 
        }, 
        sample_observations={
            "env_state": next_state
        }
    )

    next_action = int(np.asarray(next_action))

    state = next_state
    action = next_action 

    if (action == optimal_action(state)):
        count += 1
        
print(count/n_steps)


100%|██████████| 10000/10000 [00:05<00:00, 1818.48it/s]

0.8785


In [ ]:
# rl.sample(
#         update_observations={
#             "env_state": next_state, 
#             "action": action, 
#             "reward": reward, 
#             "terminal": False, 
#             "epsilon_min": np.float32(1.0),
#         }, 
#         sample_observations={
#             "env_state": next_state
#         }
#     )

In [ ]:
rl._agent.__dict__

{'obs_space_shape': (2,),
 'act_space_size': 5,
 'init': <PjitFunction of functools.partial(<function DQN.init at 0x000002606FA58CC0>, obs_space_shape=(2,), q_network=QNetwork(
     # attributes
     n_actions = 5
 ), optimizer=GradientTransformationExtraArgs(init=<function chain.<locals>.init_fn at 0x000002601FA0A980>, update=<function chain.<locals>.update_fn at 0x000002601FA08EA0>), er=ExperienceReplay(init=<PjitFunction of <function experience_replay.<locals>.init at 0x000002601FA09580>>, append=<PjitFunction of <function experience_replay.<locals>.append at 0x000002601FA0BC40>>, sample=<PjitFunction of <function experience_replay.<locals>.sample at 0x000002601FA09BC0>>, is_ready=<PjitFunction of <function experience_replay.<locals>.is_ready at 0x000002601FA08540>>), epsilon=1.0)>,
 'update': <PjitFunction of functools.partial(<function DQN.update at 0x000002606FA58E00>, step_fn=functools.partial(<function gradient_step at 0x000002606FA393A0>, optimizer=GradientTransformationExtraA